In [0]:
# libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
# Parameters
%run /Workspace/Users/arunbabuprakash@gmail.com/E_Comm/Utilities

# Brands

In [0]:
# Read Brands table
df_brands = spark.table(f"{catalog_name}.{bronze}.brz_brands")
df_brands.show(10)

In [0]:
df_brands.count()

In [0]:
df_brands.describe().show()

In [0]:
# quick Analysis
print("brand_code unique count",df_brands.select('brand_code').distinct().count())
print("brand_name unique count",df_brands.select('brand_name').distinct().count())
print("category_code unique count",df_brands.select('category_code').distinct().count())

print("sample of brand_code unique values")
print("-"*70)
df_brands.groupBy('brand_code').count().orderBy('count', ascending=False).show(10)
print("*"*70)
print("sample of brand_name unique values")
print("-"*70)
df_brands.groupBy("brand_name").count().orderBy('count', ascending=False).show(10)
print("*"*70)
print("sample of category_code unique values")
print("-"*70)
df_brands.groupBy('category_code').count().orderBy('count', ascending=False).show(10)
print("*"*70)



In [0]:
# trim columns
for column_name in df_brands.columns[:3]:
    df_brands = df_brands.withColumn(column_name,
                     trim(col(column_name)))


In [0]:
# remove special characters
for column_name in df_brands.columns[:3]:
    df_brands = df_brands.withColumn(column_name,
                                     regexp_replace(col(column_name), '[^a-zA-Z0-9]',''))

In [0]:
df_brands.show()

In [0]:
df_brands.select("category_code").distinct().show()

In [0]:
# replace category_code
category_code_replace = {
    'BOOKS' : 'BKS',
    'GROCERY' : 'GRCY',
    'TOYS' : 'TOY'
}
df_brands = df_brands.replace(category_code_replace, 'category_code')
df_brands.select('category_code').distinct().show()

In [0]:
# write to silver layer
df_brands.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.{silver}.slv_brands")



# Category

In [0]:
# Read category table from bronze layer
df_category = spark.table(f"{catalog_name}.{bronze}.brz_category")
df_category.show(10)


In [0]:
# Checking duplicates
df_category.groupBy("category_code").count().filter(col("count") > 1).show()

In [0]:
# Drop duplicates
df_category = df_category.dropDuplicates(["category_code"])


In [0]:
# Convert to uppercase
df_category = df_category.withColumn("category_code",upper(col("category_code")))


In [0]:
df_category.display()

In [0]:
# write to silver layer
df_category.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.{silver}.slv_category")


# Customers

In [0]:
# Read customers tabel from bronze layer
df_customers = spark.table(f"{catalog_name}.{bronze}.brz_customers")
df_customers.show()

In [0]:
# Checking for null values in customer_id column and droping them
print("Null values in customer_id column: ", df_customers.filter(col('customer_id').isNull()).count())
df_customers = df_customers.dropna(subset='customer_id')


In [0]:
# Checking for null values in phone column and replacing them
print("phone column null values count",df_customers.filter(col('phone').isNull()).count())
print("Replacing with some other value")
df_customers = df_customers.fillna("NA",subset='phone')

In [0]:
# replace .0 with empty string in phone column
df_customers = df_customers.withColumn("phone",regexp_replace("phone",r"\.0$",""))

In [0]:
# write to silver layer
df_customers.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.{silver}.slv_customers")


# Date

In [0]:
# Read date table from bronze layer
df_date = spark.table(f"{catalog_name}.{bronze}.brz_date")
display(df_bronze.limit(5))

In [0]:
display(df_date.describe())

In [0]:
df_date.printSchema()

In [0]:
# Change data type
df_date = df_date.withColumn("date", to_date(col("date"),"dd-MM-yyyy"))

In [0]:
# Check for duplicates and remove
df_date.groupBy("date").count().filter(col("count") > 1).show()
df_date = df_date.dropDuplicates(["date"])
print("Duplicates removed")

In [0]:
# Change day name to title case
df_date = df_date.withColumn('day_name',initcap(col('day_name')))

In [0]:
# Change week of year to positive
df_date = df_date.withColumn('week_of_year',abs(col('week_of_year')))


In [0]:
# Change data type
df_date = df_date.withColumn('week_of_year',col('week_of_year').cast('integer').cast('string'))


In [0]:
df_date.printSchema()

In [0]:
# Change quarter and week of year to string format
df_date = df_date.withColumn('quarter',concat_ws("",concat(lit('Q'),col("quarter"),lit('-'),col("year"))))
df_date = df_date.withColumn('week_of_year',concat_ws("",concat(lit('Week'),col("week_of_year"),lit('-'),col("year"))))

In [0]:
# Rename week column
df_date = df_date.withColumnRenamed("week_of_year","week")

In [0]:
# Write to silver label
df_date.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.{silver}.slv_date")

# Products

In [0]:
# Read Products tabel from bronze
df_products = spark.read.table(f"{catalog_name}.{bronze}.brz_products")
df_products.show(3)

In [0]:
df_bronze.groupBy("product_id").count().filter(col("count") > 1).show()


In [0]:
df_bronze.filter(col("product_id").isNull()).count()

In [0]:
df_bronze.select(col("weight_grams")).show(5,truncate = True)

In [0]:
df_bronze = df_bronze.withColumn("weight_grams",regexp_replace(col("weight_grams"), "g","").cast(IntegerType()))

In [0]:
df_bronze = df_bronze.withColumn("length_cm",
                    round(regexp_replace(col('length_cm'),',','.').cast(FloatType()),2))

In [0]:
df_bronze = df_bronze.withColumn('category_code',upper(col('category_code'))).withColumn('brand_code',upper(col('brand_code')))

In [0]:
df_bronze.groupBy('material').count().show()

In [0]:
df_bronze = df_bronze.withColumn('material',
                     when(col("material") == "Coton" , "Cotton").\
                         when(col("material") == "Alumium" , "Aluminum").\
                         when(col("material") == "Ruber" , "Rubber").\
                         otherwise(col("material")))

In [0]:
df_bronze.schema

In [0]:
df_bronze.filter(col('rating_count')<0).select("rating_count").show()

In [0]:
df_bronze = df_bronze.withColumn('rating_count',
                     when(col("rating_count").isNotNull(), abs(col("rating_count"))).otherwise(lit(0)).cast(IntegerType()))

In [0]:
df_bronze = df_bronze.withColumn("width_cm",round(col("width_cm").cast(FloatType()), 2)) \
                .withColumn("height_cm",round(col("height_cm").cast(FloatType()), 2))

In [0]:
df_bronze.limit(5).show()

In [0]:
df_bronze.dtypes

In [0]:
display(df_bronze.limit(5))

In [0]:
df_bronze.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.{silver}.slv_products")

# Order ideams  Fact Table

In [0]:
spark.sql(f"SHOW TABLES IN {catalog_name}.{bronze}").show()

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.{bronze}.brz_order_items")
display(df_bronze.limit(5))

In [0]:
df_bronze.groupBy('order_id','item_seq').count().filter(col('count')>1).show()

In [0]:
df_bronze = df_bronze.dropDuplicates(['order_id','item_seq'])

In [0]:
df_bronze.groupBy('quantity').count().show()

In [0]:
df_bronze = df_bronze.withColumn('quantity',when(col("quantity") == "Two","2").otherwise(col("quantity")).cast("int"))

In [0]:
df_bronze.groupBy('unit_price_currency').count().show()

In [0]:
df_bronze.groupBy("unit_price_currency").count()

In [0]:
df_bronze = df_bronze.withColumn('unit_price',regexp_replace("unit_price", "[$]", "").cast("double"))


In [0]:
df_bronze.groupBy('discount_pct').count().show()

In [0]:
df_bronze = df_bronze.withColumn('discount_pct',regexp_replace("discount_pct", "%", "").cast("double"))

In [0]:
df_bronze = df_bronze.withColumn('coupon_code',lower(trim(col('coupon_code'))))

In [0]:
df_bronze.groupBy('channel').count().show()

In [0]:
df_bronze = df_bronze.withColumn('channel',\
            when(col('channel') == 'web', 'Website').\
            when(col('channel') == 'app', 'Mobile').\
            otherwise(col('channel')))

In [0]:
df_bronze.groupBy('order_ts').count().show()

In [0]:
df_bronze = df_bronze.withColumn('dt',to_date(col('dt'),'yyyy-MM-dd'))

df_bronze = df_bronze.withColumn('order_ts',\
  coalesce(\
    to_timestamp(col('order_ts'),'yyyy-MM-dd HH:mm:ss'),\
    to_timestamp(col('order_ts'),'dd-MM-yyyy HH:mm')))

df_bronze = df_bronze.withColumn('item_seq',col("item_seq").cast("int"))

df_bronze = df_bronze.withColumn('tax_amount',col('tax_amount').cast('double'))

In [0]:
display(df_bronze.limit(5))

In [0]:
df_bronze.printSchema()

In [0]:
df_bronze.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.{silver}.slv_order_items")